<a href="https://colab.research.google.com/github/blueartseat610/data-science-assignment-task-4-/blob/main/Task4_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
!pip install pyspark

In [8]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("Tutorial4_ETL") \
    .getOrCreate()

print("Spark session started")

Spark session started


**PART 2 — Create the required tables**

**Load the dataset**

In [10]:
movies_df = spark.read.csv("movies.csv", header=True, inferSchema=True)
ratings_df = spark.read.csv("ratings.csv", header=True, inferSchema=True)

movies_df = movies_df.withColumnRenamed("movieId", "movie_id")
ratings_df = ratings_df.withColumnRenamed("userId", "user_id") \
                       .withColumnRenamed("movieId", "movie_id")

print("movies table")
movies_df.show(5, truncate=False)

print("ratings table")
ratings_df.show(5, truncate=False)

movies table
+--------+----------------------------------+-------------------------------------------+
|movie_id|title                             |genres                                     |
+--------+----------------------------------+-------------------------------------------+
|1       |Toy Story (1995)                  |Adventure|Animation|Children|Comedy|Fantasy|
|2       |Jumanji (1995)                    |Adventure|Children|Fantasy                 |
|3       |Grumpier Old Men (1995)           |Comedy|Romance                             |
|4       |Waiting to Exhale (1995)          |Comedy|Drama|Romance                       |
|5       |Father of the Bride Part II (1995)|Comedy                                     |
+--------+----------------------------------+-------------------------------------------+
only showing top 5 rows
ratings table
+-------+--------+------+---------+
|user_id|movie_id|rating|timestamp|
+-------+--------+------+---------+
|1      |1       |4.0   |964982

**Create the user table**

In [11]:
user_df = ratings_df.select("user_id").distinct().orderBy("user_id")

print("user table")
user_df.show(10)

user table
+-------+
|user_id|
+-------+
|      1|
|      2|
|      3|
|      4|
|      5|
|      6|
|      7|
|      8|
|      9|
|     10|
+-------+
only showing top 10 rows


**Create the user profiles table**

In [12]:
import pandas as pd
import numpy as np

user_pd = user_df.toPandas()

np.random.seed(42)

user_profiles_pd = pd.DataFrame({
    "user_id": user_pd["user_id"],
    "age": np.random.randint(18, 60, len(user_pd)),
    "gender": np.random.choice(["M", "F"], len(user_pd)),
    "occupation": np.random.choice(
        ["Engineer", "Designer", "Student", "Manager", "Teacher", "Doctor"],
        len(user_pd)
    )
})

# Match tutorial sample exactly for user_id 1-5 if present
sample = {
    1: [25, "M", "Engineer"],
    2: [34, "F", "Designer"],
    3: [19, "M", "Student"],
    4: [45, "F", "Manager"],
    5: [22, "M", "Student"]
}

for uid, values in sample.items():
    if uid in user_profiles_pd["user_id"].values:
        user_profiles_pd.loc[user_profiles_pd["user_id"] == uid, ["age", "gender", "occupation"]] = values

user_profiles_df = spark.createDataFrame(user_profiles_pd)

print("user_profiles table")
user_profiles_df.show(10)

user_profiles table
+-------+---+------+----------+
|user_id|age|gender|occupation|
+-------+---+------+----------+
|      1| 25|     M|  Engineer|
|      2| 34|     F|  Designer|
|      3| 19|     M|   Student|
|      4| 45|     F|   Manager|
|      5| 22|     M|   Student|
|      6| 56|     F|   Teacher|
|      7| 36|     M|    Doctor|
|      8| 40|     M|  Designer|
|      9| 28|     M|  Engineer|
|     10| 28|     F|  Engineer|
+-------+---+------+----------+
only showing top 10 rows


**Task 1**

**Find which Age Group provides the highest average ratings for each Movie Category**

In [13]:
user_profiles_df = user_profiles_df.withColumn(
    "age_group",
    F.when(F.col("age") < 25, "Under 25")
     .when((F.col("age") >= 25) & (F.col("age") <= 34), "25-34")
     .when((F.col("age") >= 35) & (F.col("age") <= 44), "35-44")
     .otherwise("45+")
)

user_profiles_df.show(10)

+-------+---+------+----------+---------+
|user_id|age|gender|occupation|age_group|
+-------+---+------+----------+---------+
|      1| 25|     M|  Engineer|    25-34|
|      2| 34|     F|  Designer|    25-34|
|      3| 19|     M|   Student| Under 25|
|      4| 45|     F|   Manager|      45+|
|      5| 22|     M|   Student| Under 25|
|      6| 56|     F|   Teacher|      45+|
|      7| 36|     M|    Doctor|    35-44|
|      8| 40|     M|  Designer|    35-44|
|      9| 28|     M|  Engineer|    25-34|
|     10| 28|     F|  Engineer|    25-34|
+-------+---+------+----------+---------+
only showing top 10 rows


**Create Movie Category**

In [14]:
movies_df = movies_df.withColumn(
    "category",
    F.split(F.col("genres"), "\\|").getItem(0)
)

movies_df.select("movie_id", "title", "genres", "category").show(10, truncate=False)

+--------+----------------------------------+-------------------------------------------+---------+
|movie_id|title                             |genres                                     |category |
+--------+----------------------------------+-------------------------------------------+---------+
|1       |Toy Story (1995)                  |Adventure|Animation|Children|Comedy|Fantasy|Adventure|
|2       |Jumanji (1995)                    |Adventure|Children|Fantasy                 |Adventure|
|3       |Grumpier Old Men (1995)           |Comedy|Romance                             |Comedy   |
|4       |Waiting to Exhale (1995)          |Comedy|Drama|Romance                       |Comedy   |
|5       |Father of the Bride Part II (1995)|Comedy                                     |Comedy   |
|6       |Heat (1995)                       |Action|Crime|Thriller                      |Action   |
|7       |Sabrina (1995)                    |Comedy|Romance                             |Comedy   |


**Use join**

In [15]:
task1_joined_df = ratings_df.join(user_df, on="user_id", how="inner") \
                            .join(user_profiles_df, on="user_id", how="inner") \
                            .join(movies_df, on="movie_id", how="inner")

task1_joined_df.select(
    "user_id", "movie_id", "rating", "title", "category", "age_group"
).show(10, truncate=False)

+-------+--------+------+------------------------------------+---------+---------+
|user_id|movie_id|rating|title                               |category |age_group|
+-------+--------+------+------------------------------------+---------+---------+
|1      |5060    |5.0   |M*A*S*H (a.k.a. MASH) (1970)        |Comedy   |25-34    |
|1      |4006    |4.0   |Transformers: The Movie (1986)      |Adventure|25-34    |
|1      |3809    |4.0   |What About Bob? (1991)              |Comedy   |25-34    |
|1      |3793    |5.0   |X-Men (2000)                        |Action   |25-34    |
|1      |3744    |4.0   |Shaft (2000)                        |Action   |25-34    |
|1      |3740    |4.0   |Big Trouble in Little China (1986)  |Action   |25-34    |
|1      |3729    |5.0   |Shaft (1971)                        |Action   |25-34    |
|1      |3703    |5.0   |Road Warrior, The (Mad Max 2) (1981)|Action   |25-34    |
|1      |3702    |5.0   |Mad Max (1979)                      |Action   |25-34    |
|1  

**Use gropu by and mean**

In [16]:
task1_avg_df = task1_joined_df.groupBy("category", "age_group") \
    .agg(F.mean("rating").alias("avg_rating")) \
    .orderBy("category", F.desc("avg_rating"))

task1_avg_df.show(50, truncate=False)

+------------------+---------+------------------+
|category          |age_group|avg_rating        |
+------------------+---------+------------------+
|(no genres listed)|45+      |3.8181818181818183|
|(no genres listed)|25-34    |3.4375            |
|(no genres listed)|Under 25 |3.3               |
|(no genres listed)|35-44    |3.0               |
|Action            |25-34    |3.538630229419703 |
|Action            |Under 25 |3.4957710741471666|
|Action            |45+      |3.43041147392606  |
|Action            |35-44    |3.3811240253052817|
|Adventure         |25-34    |3.6058909444985394|
|Adventure         |Under 25 |3.5897321428571427|
|Adventure         |45+      |3.5630714879467997|
|Adventure         |35-44    |3.526081424936387 |
|Animation         |25-34    |3.661634103019538 |
|Animation         |45+      |3.6486820428336078|
|Animation         |35-44    |3.4204980842911876|
|Animation         |Under 25 |3.378             |
|Children          |45+      |3.19211324570273  |


**find highest average rating for each Movie Category**

In [17]:
window_spec = Window.partitionBy("category").orderBy(F.desc("avg_rating"))

task1_result_df = task1_avg_df.withColumn(
    "rank",
    F.row_number().over(window_spec)
).filter(F.col("rank") == 1).drop("rank")

task1_result_df.show(50, truncate=False)

+------------------+---------+------------------+
|category          |age_group|avg_rating        |
+------------------+---------+------------------+
|(no genres listed)|45+      |3.8181818181818183|
|Action            |25-34    |3.538630229419703 |
|Adventure         |25-34    |3.6058909444985394|
|Animation         |25-34    |3.661634103019538 |
|Children          |45+      |3.19211324570273  |
|Comedy            |25-34    |3.4640604090995986|
|Crime             |25-34    |3.9280238500851787|
|Documentary       |Under 25 |3.8876404494382024|
|Drama             |25-34    |3.6672115070284406|
|Fantasy           |Under 25 |3.611111111111111 |
|Film-Noir         |35-44    |4.120689655172414 |
|Horror            |25-34    |3.190156599552573 |
|Musical           |Under 25 |4.125             |
|Mystery           |25-34    |3.911042944785276 |
|Romance           |25-34    |3.5               |
|Sci-Fi            |25-34    |3.6973684210526314|
|Thriller          |Under 25 |3.6724137931034484|


In [18]:
task1_result_df.toPandas().to_csv("task1_highest_avg_rating_by_age_group.csv", index=False)
files.download("task1_highest_avg_rating_by_age_group.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Task 2 - Find Rating Efficiency**

**Extract promotions table**

In [20]:
movies_pd = movies_df.select("movie_id").toPandas()

np.random.seed(123)

promotions_pd = pd.DataFrame({
    "movie_id": movies_pd["movie_id"],
    "budget": np.random.randint(10000, 200000, len(movies_pd))
})

promotions_df = spark.createDataFrame(promotions_pd)

print("promotions table")
promotions_df.show(10)

promotions table
+--------+------+
|movie_id|budget|
+--------+------+
|       1| 25725|
|       2| 38030|
|       3| 27730|
|       4|129906|
|       5|156449|
|       6|139130|
|       7| 56203|
|       8|163313|
|       9| 75632|
|      10|196481|
+--------+------+
only showing top 10 rows


**create avg_ratings logic**

In [21]:
avg_ratings_df = ratings_df.groupBy("movie_id") \
    .agg(F.mean("rating").alias("avg_rating"))

avg_ratings_df.show(10)

+--------+------------------+
|movie_id|        avg_rating|
+--------+------------------+
|    1580| 3.487878787878788|
|    2366|              3.64|
|    3175|              3.58|
|    1088| 3.369047619047619|
|   32460|              4.25|
|   44022| 3.217391304347826|
|   96488|              4.25|
|    1238| 4.055555555555555|
|    1342|               2.5|
|    1591|2.6346153846153846|
+--------+------------------+
only showing top 10 rows


**perform LEFT JOIN starting from promotions**

In [22]:
task2_joined_df = promotions_df.join(
    avg_ratings_df,
    on="movie_id",
    how="left"
)

task2_joined_df.show(10)

+--------+------+------------------+
|movie_id|budget|        avg_rating|
+--------+------+------------------+
|       1| 25725|3.9209302325581397|
|       2| 38030|3.4318181818181817|
|       3| 27730|3.2596153846153846|
|       4|129906| 2.357142857142857|
|       5|156449|3.0714285714285716|
|       6|139130| 3.946078431372549|
|       7| 56203| 3.185185185185185|
|       8|163313|             2.875|
|       9| 75632|             3.125|
|      10|196481| 3.496212121212121|
+--------+------+------------------+
only showing top 10 rows


**use F.coalesce()**

In [23]:
task2_joined_df = task2_joined_df.withColumn(
    "avg_rating",
    F.coalesce(F.col("avg_rating"), F.lit(0.0))
)

task2_joined_df.show(10)

+--------+------+------------------+
|movie_id|budget|        avg_rating|
+--------+------+------------------+
|       1| 25725|3.9209302325581397|
|       2| 38030|3.4318181818181817|
|       3| 27730|3.2596153846153846|
|       4|129906| 2.357142857142857|
|       5|156449|3.0714285714285716|
|       6|139130| 3.946078431372549|
|       7| 56203| 3.185185185185185|
|       8|163313|             2.875|
|       9| 75632|             3.125|
|      10|196481| 3.496212121212121|
+--------+------+------------------+
only showing top 10 rows


**calculate Efficiency**

In [24]:
task2_result_df = task2_joined_df.withColumn(
    "efficiency",
    F.col("avg_rating") / (F.col("budget") / F.lit(10000))
)

In [25]:
task2_result_df = task2_result_df.join(
    movies_df.select("movie_id", "title"),
    on="movie_id",
    how="left"
).select(
    "movie_id", "title", "budget", "avg_rating", "efficiency"
)

**Load: sort by highest Efficiency**

In [26]:
task2_result_df = task2_result_df.orderBy(F.desc("efficiency"))

task2_result_df.show(20, truncate=False)

+--------+---------------------------------------------------------------------+------+-----------------+------------------+
|movie_id|title                                                                |budget|avg_rating       |efficiency        |
+--------+---------------------------------------------------------------------+------+-----------------+------------------+
|172909  |Cheburashka (1971)                                                   |10532 |5.0              |4.74743638435245  |
|172793  |Vovka in the Kingdom of Far Far Away (1965)                          |10706 |5.0              |4.670278348589576 |
|142444  |The Editor (2015)                                                    |11053 |5.0              |4.523658735185018 |
|57183   |Like Stars on Earth (Taare Zameen Par) (2007)                        |10071 |4.5              |4.468275245755138 |
|188189  |Sorry to Bother You (2018)                                           |10335 |4.5              |4.354136429608127 |


**Load Output**

In [27]:
task2_result_df.toPandas().to_csv("task2_rating_efficiency.csv", index=False)
files.download("task2_rating_efficiency.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>